In [13]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, types # Export to DataBase

# read XPT file
df_questionaire_bp = pd.read_sas("../data/questionaire_data/BPQ_L.xpt", format="xport", encoding="utf-8")

# show top 5 rows 
df_questionaire_bp.head()

,SEQN,BPQ020,BPQ030,BPQ150,BPQ080,BPQ101D
0,130378.0,1.0,1.0,1.0,2.0,2.0
1,130379.0,1.0,1.0,1.0,2.0,2.0
2,130380.0,2.0,NaN,NaN,1.0,1.0
3,130384.0,2.0,NaN,NaN,2.0,2.0
4,130385.0,2.0,NaN,NaN,2.0,2.0


In [14]:
print("Blood Pressure Questionaire Data Info:")
df_questionaire_bp.info() 

Blood Pressure Questionaire Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8501 entries, 0 to 8500
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   SEQN     8501 non-null   float64
 1   BPQ020   8498 non-null   float64
 2   BPQ030   2968 non-null   float64
 3   BPQ150   2969 non-null   float64
 4   BPQ080   8498 non-null   float64
 5   BPQ101D  8498 non-null   float64
dtypes: float64(6)
memory usage: 398.6 KB


In [15]:
# Select and rename essential columns
    # We'll keep SEQN for merging

columns_to_keep_and_rename_bpq = {
    'SEQN': 'Participant_ID',
    'BPQ020': 'Ever_Had_Hypertension',
    'BPQ030': 'Hypertension_Told_Twice',
    'BPQ150': 'Currently_Taking_BP_Meds',
    'BPQ080': 'Ever_Had_High_Cholesterol',
    'BPQ101D': 'Currently_Taking_Lower_Blood_Cholesterol_Meds'
}

df_questionaire_bp = df_questionaire_bp[list(columns_to_keep_and_rename_bpq.keys())].copy() # list(...) transfers dicts into lists, which then can be worked in dataframe. 
df_questionaire_bp.rename(columns=columns_to_keep_and_rename_bpq, inplace=True) 

print("--- Selected and Renamed Blood Pressure Questionaire Data Info ---")
df_questionaire_bp.info()

--- Selected and Renamed Blood Pressure Questionaire Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8501 entries, 0 to 8500
Data columns (total 6 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Participant_ID                                 8501 non-null   float64
 1   Ever_Had_Hypertension                          8498 non-null   float64
 2   Hypertension_Told_Twice                        2968 non-null   float64
 3   Currently_Taking_BP_Meds                       2969 non-null   float64
 4   Ever_Had_High_Cholesterol                      8498 non-null   float64
 5   Currently_Taking_Lower_Blood_Cholesterol_Meds  8498 non-null   float64
dtypes: float64(6)
memory usage: 398.6 KB


In [18]:
df_questionaire_bp = df_questionaire_bp.convert_dtypes() 
df_questionaire_bp # 1 = ‘Yes’； 2 = ‘No’

,Participant_ID,Ever_Had_Hypertension,Hypertension_Told_Twice,Currently_Taking_BP_Meds,Ever_Had_High_Cholesterol,Currently_Taking_Lower_Blood_Cholesterol_Meds
0,130378,1,1,1,2,2
1,130379,1,1,1,2,2
2,130380,2,<NA>,<NA>,1,1
3,130384,2,<NA>,<NA>,2,2
4,130385,2,<NA>,<NA>,2,2
...,...,...,...,...,...,...
8496,142305,1,1,1,1,1
8497,142307,2,<NA>,<NA>,1,1
8498,142308,2,<NA>,<NA>,2,2
8499,142309,2,<NA>,<NA>,2,2


In [19]:
# export as csv
file_path = "../data/questionaire_data/cleaned_blood_pressure_questionaire_data.csv" 

try:
    df_questionaire_bp.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

DataFrame successfully saved to: ../data/questionaire_data/cleaned_blood_pressure_questionaire_data.csv


In [20]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

In [21]:
df_questionaire_bp.to_sql(name = 'blood_pressure_questionaire_data', con=engine, schema='capstone_group_3',if_exists='replace',index=False)

501